# HB-PINNs: A Heat-Balance Physics-Informed Framework for Building Energy Loading Forecasting with Limited Data

**Paper:** Zhang, M., Li, Z., Yu, Z. (2026). *HB-PINNs: A Heat-Balance Physics-Informed Framework for Building Energy Loading Forecasting with Limited Data.* Preprint, University of Liverpool. https://ssrn.com/abstract=6412515

**Carpeta origen:** `PINNs/2. Termidinamica y cinetica/HB-PINNs A Heat-Balance Physics-Informed Framework for Building.pdf`

## Como se usan las PINNs en este paper

A diferencia de la mayoria de PINNs (que residualizan una EDP continua), aqui la "fisica" es la **ecuacion algebraica de balance termico** de una zona de edificio (Eq. 1):

$$Q_{sys}(t)=c_z\frac{dT_z}{dt}(t)-\sum_i Q_i(t)-\sum_i h_iA_i\big(T_{S_i}(t)-T_z(t)\big)-\dot m_{inf}(t)c_p\big(T_\infty(t)-T_z(t)\big)$$

El enfoque **HB-PINNs** (Fig. 3) es una **formulacion inversa fisicamente informada**: la red NO predice directamente la demanda de calefaccion $Q_{sys}$. En su lugar, a partir de 7 variables observables (setpoint $T_t$, $\Delta T=T_t-T_{t-1}$, $\Delta T_{in}=T_t-T_{out}$, $T_{t-1}$, ganancias solares $Q^{irr}_t$, ganancias internas $Q^{rest}_t$, demanda previa $Q_{sys(t-1)}$), la red predice **variables fisicas latentes no medibles**: temperaturas superficiales interiores $T_{S_i}$, coeficientes de conveccion $h_i$, y el caudal masico de infiltracion $\dot m_{inf}$. Estas variables latentes se sustituyen en la Eq. (1) para **reconstruir** $\hat Q_{sys}$, y la perdida es

$$Loss = MSE(Q_{sys},\hat Q_{sys})$$

Esta formulacion inversa fuerza a la red a descubrir variables fisicas consistentes con la ecuacion de balance, en vez de simplemente memorizar el mapeo entrada-salida. El paper reporta que HB-PINNs supera consistentemente a un modelo RC simplificado, a una red neuronal pura, y a RC-PINNs (que combinan una red con un modelo RC de baja fidelidad), especialmente con pocos datos de entrenamiento (Tabla 2: HB-PINNs alcanza $R^2$ promedio de 94.99% frente a 88.69% de RC-PINNs).

Este cuaderno reproduce fielmente: la generacion de un escenario termico sintetico de una zona (analogo a la simulacion EnergyPlus del paper), las 7 caracteristicas de entrada (Fig. 3), la red que predice variables latentes $(T_{S_1}, h_1, \dot m_{inf})$, y la reconstruccion de $\hat Q_{sys}$ via la ecuacion de balance termico con perdida $MSE(Q_{sys},\hat Q_{sys})$.

## Repositorio publico de referencia

El PDF (preprint SSRN) no incluye un repositorio de codigo propio, ni se encontro uno publico especifico para HB-PINNs al buscar en GitHub. Como referencia general de PINNs aplicadas a dinamica termica de edificios (el mismo dominio de aplicacion), se usa el repositorio de referencia canonico de PINNs:

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Generacion de un escenario termico sintetico (analogo a la simulacion EnergyPlus del paper, Fig. 4)

Se simula una zona unica (analoga al prototipo residencial UK del paper: area ~18.9 m^2, capacitancia termica $c_z$, transmitancia envolvente) con un controlador termostatico que seria la escena de 'verdad' (ground truth) generadora de $Q_{sys}(t)$.

In [ ]:
n_hours = 240  # 10 dias horarios
dt = 1.0
hours = np.arange(n_hours)

c_z = 5.0e6       # capacitancia termica de la zona [J/K] (normalizada; da una constante de
                  # tiempo termica de un par de horas, fisicamente razonable y numericamente estable)
h1_true = 8.0     # coeficiente de conveccion interior [W/m^2K]
A1 = 60.0         # area interior total [m^2]
m_inf_true = 0.03 # caudal masico de infiltracion [kg/s]
cp = 1006.0       # calor especifico del aire [J/kgK]
Q_max = 2000.0    # potencia maxima de calefaccion [W]

# Escenario 'Comfort': setpoint constante 21 C durante el dia, 18 C de noche
hour_of_day = hours % 24
T_setpoint = np.where((hour_of_day >= 6) & (hour_of_day <= 22), 21.0, 18.0)
T_out = 8.0 + 5.0 * np.sin(2 * np.pi * (hours - 6) / 24)  # temperatura exterior diaria
Q_irr = np.clip(300 * np.sin(2 * np.pi * (hour_of_day - 6) / 24), 0, None)  # ganancia solar
Q_rest = 100.0 + 50.0 * (np.sin(2 * np.pi * (hour_of_day - 18) / 24) > 0.3)  # ganancias internas

T_z = np.zeros(n_hours)
Q_sys = np.zeros(n_hours)
T_S1 = np.zeros(n_hours)
T_z[0] = 18.0
for k in range(n_hours - 1):
    # controlador proporcional simple hacia el setpoint, saturado a [0, Q_max]
    Q_sys[k] = np.clip(400.0 * (T_setpoint[k] - T_z[k]), 0, Q_max)
    T_S1[k] = 0.5 * (T_z[k] + T_out[k])  # temperatura superficial interior aproximada
    dTdt = (Q_sys[k] + Q_irr[k] + Q_rest[k]
            - h1_true * A1 * (T_z[k] - T_S1[k])
            - m_inf_true * cp * (T_z[k] - T_out[k])) / c_z
    T_z[k + 1] = T_z[k] + dTdt * dt * 3600
Q_sys[-1] = Q_sys[-2]
T_S1[-1] = T_S1[-2]

plt.figure(figsize=(10, 3))
plt.plot(hours, Q_sys, label='Q_sys (demanda de calefaccion)')
plt.plot(hours, T_z * 50, '--', alpha=0.5, label='T_z (escalada x50 para visualizar)')
plt.xlabel('Hora'); plt.legend(); plt.title("Escenario termico sintetico ('Comfort')")
plt.show()

## 2. Caracteristicas de entrada (Fig. 3: 7 variables observables) y red HB-PINN

In [ ]:
delta_T = np.diff(T_setpoint, prepend=T_setpoint[0])
delta_T_in = T_setpoint - T_out
T_prev = np.roll(T_setpoint, 1); T_prev[0] = T_setpoint[0]
Q_sys_prev = np.roll(Q_sys, 1); Q_sys_prev[0] = 0.0

features = np.stack([T_setpoint, delta_T, delta_T_in, T_prev, Q_irr, Q_rest, Q_sys_prev], axis=1)
feat_mean, feat_std = features.mean(0), features.std(0) + 1e-8
features_norm = (features - feat_mean) / feat_std

dTz_dt = np.gradient(T_z, dt * 3600)  # dT_z/dt observado (diferencias finitas)

n_train = int(0.6 * n_hours)  # 60% de los datos, uno de los ratios evaluados en la Tabla 2 del paper
X = torch.tensor(features_norm, dtype=torch.float32, device=device)
Qsys_t = torch.tensor(Q_sys, dtype=torch.float32, device=device).view(-1, 1)
Tz_t = torch.tensor(T_z, dtype=torch.float32, device=device).view(-1, 1)
Tout_t = torch.tensor(T_out, dtype=torch.float32, device=device).view(-1, 1)
Qgains_t = torch.tensor(Q_irr + Q_rest, dtype=torch.float32, device=device).view(-1, 1)
dTzdt_t = torch.tensor(dTz_dt, dtype=torch.float32, device=device).view(-1, 1)


class HBPINN(nn.Module):
    """Predice variables fisicas latentes: T_S1, h1, m_inf (Fig. 3)."""
    def __init__(self, n_hidden=3, n_neurons=32):
        super().__init__()
        layers = [nn.Linear(7, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 3)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        out = self.net(x)
        T_S1 = out[:, 0:1] * 10 + 15    # rango fisico razonable de temperatura superficial
        h1 = torch.nn.functional.softplus(out[:, 1:2]) + 1.0   # h1 > 0
        m_inf = torch.nn.functional.softplus(out[:, 2:3]) * 0.1  # m_inf >= 0
        return T_S1, h1, m_inf

## 3. Reconstruccion de $\hat Q_{sys}$ via la ecuacion de balance termico (Eq. 1) y perdida

In [ ]:
def reconstruct_Qsys(model, X, Tz, Tout, Qgains, dTzdt):
    T_S1, h1, m_inf = model(X)
    Q_hat = (c_z * dTzdt - Qgains
             - h1 * A1 * (T_S1 - Tz)
             - m_inf * cp * (Tout - Tz))
    return Q_hat, T_S1, h1, m_inf


model = HBPINN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []

for epoch in range(3000):
    optimizer.zero_grad()
    Q_hat, *_ = reconstruct_Qsys(model, X[:n_train], Tz_t[:n_train], Tout_t[:n_train],
                                  Qgains_t[:n_train], dTzdt_t[:n_train])
    loss = torch.mean((Q_hat - Qsys_t[:n_train])**2)
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if epoch % 500 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e}')

## 4. Evaluacion: $R^2$ en datos de entrenamiento y de prueba (cf. Tabla 2 del paper)

In [ ]:
def r_squared(y_true, y_pred):
    ss_res = torch.sum((y_true - y_pred)**2)
    ss_tot = torch.sum((y_true - y_true.mean())**2)
    return (1 - ss_res / ss_tot).item()


with torch.no_grad():
    Q_hat_all, T_S1_all, h1_all, minf_all = reconstruct_Qsys(model, X, Tz_t, Tout_t, Qgains_t, dTzdt_t)

r2_train = r_squared(Qsys_t[:n_train], Q_hat_all[:n_train])
r2_test = r_squared(Qsys_t[n_train:], Q_hat_all[n_train:])
print(f'R^2 entrenamiento (60% de los datos): {r2_train*100:.2f}%')
print(f'R^2 prueba (40% restante):            {r2_test*100:.2f}%')
print(f'h1 medio inferido: {h1_all.mean().item():.2f} (verdad sintetica: {h1_true})')
print(f'm_inf medio inferido: {minf_all.mean().item():.4f} (verdad sintetica: {m_inf_true})')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hours, Q_sys, label='$Q_{sys}$ (verdad)')
axes[0].plot(hours, Q_hat_all.cpu().numpy(), '--', label='$\\hat Q_{sys}$ (HB-PINN)')
axes[0].axvline(n_train, color='gray', linestyle=':', label='train/test')
axes[0].set_xlabel('Hora'); axes[0].set_ylabel('Demanda de calefaccion [W]')
axes[0].legend(); axes[0].set_title('Reconstruccion de la demanda de calefaccion')

axes[1].semilogy(history)
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Loss (escala log)')
axes[1].set_title('Convergencia del entrenamiento')
plt.tight_layout()
plt.show()